# ML Trading System - Complete Workflow

Dieses Notebook zeigt den kompletten Workflow vom Daten-Download bis zum Backtesting.

## Inhalt
1. Setup & Daten laden
2. Feature Engineering
3. Model Training
4. Backtesting
5. Evaluation & Visualisierung
6. Model Comparison

In [ ]:
# Imports
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.data_fetcher import DataFetcher
from src.data.data_cleaner import DataCleaner
from src.features.feature_engineer import FeatureEngineer
from src.models.trainer import ModelTrainer
from src.models.experiment_tracker import ExperimentTracker
from src.backtest.backtester import Backtester
from src.backtest.strategies import StrategyFactory
from src.evaluation.visualizer import Visualizer
from src.utils.config_loader import load_config

%matplotlib inline
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Setup & Daten laden

In [ ]:
# Load configuration
config = load_config('../config/config.yaml')
print("Configuration loaded")
print(f"Symbol: {config['data']['symbol']}")
print(f"Timeframe: {config['data']['timeframes'][0]}")

In [ ]:
# Fetch data
fetcher = DataFetcher('binance')
df = fetcher.fetch_ohlcv(
    symbol='BTC/USDT',
    timeframe='1h',
    start_date='2023-01-01',
    limit=1000
)

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

In [ ]:
# Clean data
cleaner = DataCleaner()
df_clean = cleaner.clean_ohlcv(df)

# Data quality report
quality = cleaner.get_data_quality_report(df_clean)
print("\nData Quality Report:")
for key, value in quality.items():
    print(f"{key}: {value}")

In [ ]:
# Plot price chart
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Price
ax1.plot(df_clean.index, df_clean['close'], linewidth=1.5)
ax1.set_title('BTC/USDT Price', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (USDT)')
ax1.grid(True, alpha=0.3)

# Volume
ax2.bar(df_clean.index, df_clean['volume'], alpha=0.7)
ax2.set_title('Volume', fontsize=14, fontweight='bold')
ax2.set_ylabel('Volume')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
# Create features
engineer = FeatureEngineer(config['features'])
df_features = engineer.create_features(df_clean)

print(f"Original columns: {len(df_clean.columns)}")
print(f"After features: {len(df_features.columns)}")
print(f"\nNew features: {len(df_features.columns) - len(df_clean.columns)}")

In [ ]:
# Create labels
df_labeled = engineer.create_labels(
    df_features,
    label_type='classification',
    horizons=[1, 5, 20],
    threshold=0.02
)

# Label distribution
target_col = 'target_1'
label_counts = df_labeled[target_col].value_counts()
print("\nLabel Distribution (1h horizon):")
print(label_counts)
print(f"\nClass balance: {label_counts.min() / label_counts.max():.2f}")

In [ ]:
# Prepare ML data
X, y, feature_names = engineer.prepare_ml_data(df_labeled, target_col='target_1')

print(f"\nML Dataset:")
print(f"Samples: {len(X)}")
print(f"Features: {len(feature_names)}")
print(f"\nFeature groups:")

groups = engineer.get_feature_groups()
for group, prefixes in groups.items():
    count = sum(1 for f in feature_names if any(p in f for p in prefixes))
    print(f"  {group}: {count} features")

## 3. Model Training

In [ ]:
# Initialize trainer
trainer = ModelTrainer(config['models'])
trainer.feature_names = feature_names

# Split data
X_train, X_val, X_test, y_train, y_val, y_test = trainer.split_data(X, y)

print(f"Train: {len(X_train)} samples")
print(f"Val:   {len(X_val)} samples")
print(f"Test:  {len(X_test)} samples")

In [ ]:
# Scale features
X_train_scaled, X_val_scaled, X_test_scaled = trainer.scale_features(
    X_train, X_val, X_test
)

print("Features scaled")
print(f"Mean: {X_train_scaled.mean().mean():.6f}")
print(f"Std: {X_train_scaled.std().mean():.6f}")

In [ ]:
# Train XGBoost model
model = trainer.train_model(
    'xgboost',
    X_train_scaled,
    y_train
)

print("Model trained!")

In [ ]:
# Evaluate on test set
metrics = trainer.evaluate(model, X_test_scaled, y_test)

print("\nTest Set Performance:")
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1 Score:  {metrics['f1']:.4f}")

print("\n" + metrics['classification_report'])

In [ ]:
# Feature importance
importance_df = trainer.get_feature_importance(model, feature_names, top_n=20)

# Plot
plt.figure(figsize=(10, 8))
plt.barh(range(len(importance_df)), importance_df['importance'])
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

importance_df

## 4. Backtesting

In [ ]:
# Generate predictions on full dataset
X_full_scaled = pd.DataFrame(
    trainer.scaler.transform(X),
    index=X.index,
    columns=X.columns
)

predictions = pd.Series(
    model.predict(X_full_scaled),
    index=X_full_scaled.index
)

print("Predictions generated")
print(f"Signal distribution:")
print(predictions.value_counts())

In [ ]:
# Create strategy
strategy = StrategyFactory.create_strategy('pure_ml', config['backtest'])

# Generate signals
signals = strategy.generate_signals(df_labeled, predictions)

print("Trading signals:")
print(f"Buy:  {(signals == 2).sum()}")
print(f"Hold: {(signals == 1).sum()}")
print(f"Sell: {(signals == 0).sum()}")

In [ ]:
# Run backtest
backtester = Backtester(config['backtest'])
results = backtester.run(df_labeled, signals)

print("\n" + "="*60)
print("BACKTEST RESULTS")
print("="*60)
print(f"Initial Capital:    ${results['initial_capital']:,.2f}")
print(f"Final Equity:       ${results['final_equity']:,.2f}")
print(f"Total Return:       {results['total_return']*100:.2f}%")
print(f"CAGR:               {results['cagr']*100:.2f}%")
print(f"")
print(f"Sharpe Ratio:       {results['sharpe_ratio']:.2f}")
print(f"Sortino Ratio:      {results['sortino_ratio']:.2f}")
print(f"Max Drawdown:       {results['max_drawdown']*100:.2f}%")
print(f"")
print(f"Total Trades:       {results['total_trades']}")
print(f"Win Rate:           {results['win_rate']*100:.2f}%")
print(f"Profit Factor:      {results['profit_factor']:.2f}")
print("="*60)

## 5. Visualization

In [ ]:
# Equity curve
equity_series = pd.Series(results['equity_curve'], index=results['equity_dates'])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(equity_series.index, equity_series.values, linewidth=2)
ax.fill_between(equity_series.index, equity_series.values, alpha=0.3)
ax.axhline(y=results['initial_capital'], color='r', linestyle='--', label='Initial Capital')
ax.set_title('Equity Curve', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Equity ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown
cumulative_max = equity_series.expanding().max()
drawdown = (equity_series - cumulative_max) / cumulative_max * 100

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(drawdown.index, drawdown.values, 0, 
                where=drawdown.values < 0, color='red', alpha=0.3)
ax.plot(drawdown.index, drawdown.values, color='red', linewidth=2)
ax.set_title('Drawdown', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown (%)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Trade analysis
trade_log = backtester.get_trade_log()

if not trade_log.empty:
    print(f"\nTrade Log: {len(trade_log)} trades")
    trade_log.head(10)
else:
    print("No trades executed")

In [ ]:
# Returns distribution
if not trade_log.empty:
    returns = trade_log['return_pct'] * 100
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax1.hist(returns, bins=30, alpha=0.7, edgecolor='black')
    ax1.axvline(returns.mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {returns.mean():.2f}%')
    ax1.axvline(0, color='black', linestyle='-', linewidth=1)
    ax1.set_xlabel('Return (%)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Returns Distribution')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Box plot
    ax2.boxplot(returns, vert=True)
    ax2.set_ylabel('Return (%)')
    ax2.set_title('Returns Box Plot')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Strategy Comparison

In [ ]:
# Compare different strategies
strategies = ['pure_ml', 'momentum_ml', 'mean_reversion_ml', 'volatility_breakout_ml']
comparison_results = {}

for strategy_name in strategies:
    print(f"\nTesting {strategy_name}...")
    
    strategy = StrategyFactory.create_strategy(strategy_name, config['backtest'])
    signals = strategy.generate_signals(df_labeled, predictions)
    
    backtester = Backtester(config['backtest'])
    results = backtester.run(df_labeled, signals)
    
    comparison_results[strategy_name] = {
        'Total Return': results['total_return'],
        'Sharpe Ratio': results['sharpe_ratio'],
        'Max Drawdown': results['max_drawdown'],
        'Win Rate': results['win_rate'],
        'Total Trades': results['total_trades']
    }

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_results).T
comparison_df['Total Return'] = comparison_df['Total Return'] * 100
comparison_df['Max Drawdown'] = comparison_df['Max Drawdown'] * 100
comparison_df['Win Rate'] = comparison_df['Win Rate'] * 100

print("\n" + "="*80)
print("STRATEGY COMPARISON")
print("="*80)
comparison_df

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total Return
comparison_df['Total Return'].plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Total Return (%)', fontweight='bold')
axes[0,0].set_ylabel('Return (%)')
axes[0,0].grid(True, alpha=0.3)

# Sharpe Ratio
comparison_df['Sharpe Ratio'].plot(kind='bar', ax=axes[0,1], color='green')
axes[0,1].set_title('Sharpe Ratio', fontweight='bold')
axes[0,1].set_ylabel('Sharpe')
axes[0,1].grid(True, alpha=0.3)

# Max Drawdown
comparison_df['Max Drawdown'].plot(kind='bar', ax=axes[1,0], color='red')
axes[1,0].set_title('Max Drawdown (%)', fontweight='bold')
axes[1,0].set_ylabel('Drawdown (%)')
axes[1,0].grid(True, alpha=0.3)

# Win Rate
comparison_df['Win Rate'].plot(kind='bar', ax=axes[1,1], color='orange')
axes[1,1].set_title('Win Rate (%)', fontweight='bold')
axes[1,1].set_ylabel('Win Rate (%)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

Dieser Workflow zeigt:
1. ✅ Daten-Download und Bereinigung
2. ✅ Feature Engineering mit 50+ Indikatoren
3. ✅ Model Training und Evaluation
4. ✅ Backtesting mit verschiedenen Strategien
5. ✅ Performance-Analyse und Visualisierung
6. ✅ Strategy Comparison

Nächste Schritte:
- Hyperparameter-Tuning für bessere Performance
- Walk-Forward Analysis für robustere Validation
- Paper Trading zum Live-Test